# Модуль 03 · Урок 3. Очистка строк и регулярные выражения

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Нормализация строк
Системы могут записывать одну категорию как `Food`, ` FOOD ` и `food`. Стандартная последовательность `str.strip().str.lower()` устраняет крайние пробелы и различия регистра; иногда нужна `str.casefold()` для международных строк. Прежде чем нормализовать данные, важно понимать, допустимо ли изменять регистр идентификаторов: артикулы `a1` и `A1` могут обозначать разные товары.

In [ ]:
import pandas as pd
labels = pd.Series(['  Sales ','SALES', None,'sales '])
print(labels.str.strip().str.lower())

## 2. Удаление символов и разбиение
`str.replace(old,new,regex=False)` выполняет буквальную замену, а `str.replace(pattern,new,regex=True)` — по регулярному выражению. `str.split('-',expand=True)` разбивает код на отдельные столбцы; `str.extract(r'(\d+)')` извлекает подстроку по шаблону. Помните, что в регулярных выражениях `.` означает любой символ, если не экранирован обратным слешем.

In [ ]:
codes = pd.Series(['BOX-012','BAG-055','bad'])
print(codes.str.extract(r'([A-Z]+)-(\d+)'))

## 3. Поиск и валидация шаблонов
`str.contains(r'...',na=False)` проверяет наличие совпадения где угодно, а `str.fullmatch(r'...',na=False)` — соответствие **всей строки**. Для проверки простого кода вида три заглавные буквы, дефис, две цифры нужен полный матч. Regex легко сделать слишком разрешительным; ограничивайте начало/конец или используйте `fullmatch`.

In [ ]:
ids = pd.Series(['ABC-12','ABC-123','xABC-12',None])
print(ids.str.fullmatch(r'[A-Z]{3}-\d{2}',na=False))

## 4. Работа с пропусками в текстовых столбцах
Методы `.str` обычно пропускают NA, а не превращают их в строку `'None'`. Применение `astype(str)` иногда даёт не то, что нужно для бизнес-логики: строковое `'nan'` может попасть в результат. Для очистки лучше сначала использовать nullable string и отдельно контролировать пропуски.

**Проверьте себя:** в чём разница между `contains` и `fullmatch`? Когда нужен `regex=False`? Почему отсутствие почты нельзя автоматически заменять строкой `'None'`?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m3_11 · category key

Добавьте `category_key`: значения category без крайних пробелов, в нижнем регистре; пропуски замените строкой `unknown`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_11: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_11 = expected_for("m3_11", df=df)


In [ ]:
# m3_11: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 6 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`, `category_key`.

```text
     item   price quantity category          email category_key
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM         food
2     b-2                3     food        b@x.ru          food
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM         food
8     C-3  ошибка     None      TOY            bad          toy
10   None       0        7     None                     unknown
15   d-4    -4.00        1      toy           None          toy
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_11", result, _expected_m3_11)

### m3_12 · normalize email

Добавьте `email_norm`: удалите крайние пробелы и переведите почту в нижний регистр; пропуски сохраните.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_12: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_12 = expected_for("m3_12", df=df)


In [ ]:
# m3_12: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 6 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`, `email_norm`.

```text
     item   price quantity category          email     email_norm
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM  a@example.com
2     b-2                3     food        b@x.ru          b@x.ru
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM  a@example.com
8     C-3  ошибка     None      TOY            bad            bad
10   None       0        7     None                              
15   d-4    -4.00        1      toy           None           None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_12", result, _expected_m3_12)

### m3_13 · extract code number

Добавьте `item_number`: извлеките цифры после дефиса из очищенного поля `item`; преобразуйте в число, ошибки сделайте пропусками.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_13: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_13 = expected_for("m3_13", df=df)


In [ ]:
# m3_13: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 6 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`, `item_number`.

```text
     item   price quantity category          email  item_number
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM          1.0
2     b-2                3     food        b@x.ru           2.0
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM          1.0
8     C-3  ошибка     None      TOY            bad          3.0
10   None       0        7     None                         NaN
15   d-4    -4.00        1      toy           None          4.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_13", result, _expected_m3_13)

### m3_14 · valid simple email

Получите логическую Series: адрес без пробелов по краям соответствует упрощённому шаблону `символы@символы.домен`, без внутренних пробелов. Пропуски → False.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_14: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_14 = expected_for("m3_14", df=df)


In [ ]:
# m3_14: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 6 значений; имя серии: `email`.

```text
11     True
2      True
3      True
8     False
10    False
15    False
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_14", result, _expected_m3_14)

### m3_15 · item prefix

Добавьте `item_prefix`: буквы перед дефисом после удаления пробелов и приведения к верхнему регистру. У отсутствующих или некорректных значений — пропуск.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_15: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_15 = expected_for("m3_15", df=df)


In [ ]:
# m3_15: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 6 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`, `item_prefix`.

```text
     item   price quantity category          email item_prefix
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM           A
2     b-2                3     food        b@x.ru            B
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM           A
8     C-3  ошибка     None      TOY            bad           C
10   None       0        7     None                       None
15   d-4    -4.00        1      toy           None           D
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_15", result, _expected_m3_15)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()